In [1]:
# ==============================================================================
# PIX2PIXHD END-TO-END PIPELINE (768x1024 Portrait) - FIX: MODERN PYTHON PATCH
# ==============================================================================
# 1. Installs modern GPU-compatible PyTorch & Dependencies (Skips if done)
# 2. Clones the Pix2PixHD repository to your home directory (Skips if done)
# 3. Prepares your dataset from 3072x4096 to 768x1024 (Skips if done)
# 4. Auto-patches fractions.gcd syntax error and launches Training safely
# 5. Generates the final Synthetic Images immediately after training completes
# ==============================================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

# Path Setup
HOME_DIR = "/home/project/26060809/RE8_PNG_Masks"
REPO_DIR = os.path.join(HOME_DIR, "pix2pixHD")
HD_DATASET_DIR = os.path.join(HOME_DIR, "palm_oil_hd")

TRAIN_A_DIR = os.path.join(HD_DATASET_DIR, "train_A")  
TRAIN_B_DIR = os.path.join(HD_DATASET_DIR, "train_B")  
TEST_A_DIR = os.path.join(HD_DATASET_DIR, "test_A")    

# ------------------------------------------------------------------------------
# STEP 1: Dependencies & Environment Setup (Conditional Bypass)
# ------------------------------------------------------------------------------
print("--- STEP 1: Checking environment & packages ---")
try:
    import torch
    import cv2
    import dominate
    print("✅ Core dependencies already installed. Skipping installation phase.")
except ImportError:
    print("Installing PyTorch (CUDA 12.1 compatibility for NVIDIA L40S)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"])
    print("Installing Pix2PixHD supporting libraries...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "dominate", "scipy", "pillow", "scikit-image", "visdom", "opencv-python"])
    import torch
    import cv2

print(f"PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

# ------------------------------------------------------------------------------
# STEP 2: Clone Pix2PixHD Repository (Conditional Bypass)
# ------------------------------------------------------------------------------
print("\n--- STEP 2: Checking Pix2PixHD repository ---")
if not os.path.exists(REPO_DIR):
    print(f"Cloning codebase to: {REPO_DIR}")
    subprocess.run(["git", "clone", "https://github.com/NVIDIA/pix2pixHD.git", REPO_DIR], check=True)
else:
    print("✅ Pix2PixHD repository already exists. Skipping clone.")

# ------------------------------------------------------------------------------
# STEP 3: Dataset Preparation (Conditional Bypass)
# ------------------------------------------------------------------------------
print("\n--- STEP 3: Checking dataset directory status ---")

dataset_ready = (
    os.path.exists(TRAIN_A_DIR) and len(os.listdir(TRAIN_A_DIR)) > 0 and
    os.path.exists(TRAIN_B_DIR) and len(os.listdir(TRAIN_B_DIR)) > 0 and
    os.path.exists(TEST_A_DIR) and len(os.listdir(TEST_A_DIR)) > 0
)

if dataset_ready:
    print(f"✅ Processed dataset found at '{HD_DATASET_DIR}'. Skipping image generation mapping.")
else:
    print("Dataset incomplete or missing. Reprocessing 768x1024 portrait targets...")
    # UPDATED: Pointed directly to your flat train directory where both images and masks live
    SRC_DATA_PATH = os.path.join(HOME_DIR, "train")

    if os.path.exists(HD_DATASET_DIR):
        shutil.rmtree(HD_DATASET_DIR)

    os.makedirs(TRAIN_A_DIR, exist_ok=True)
    os.makedirs(TRAIN_B_DIR, exist_ok=True)
    os.makedirs(TEST_A_DIR, exist_ok=True)

    def mask_to_rgb(mask):
        rgb = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
        rgb[mask == 0] = [0, 0, 0]      
        rgb[mask == 1] = [0, 255, 0]    
        rgb[mask == 2] = [150, 75, 0]   
        return rgb

    if os.path.exists(SRC_DATA_PATH):
        # UPDATED: Scans directory to discover and isolate target image objects
        all_files = os.listdir(SRC_DATA_PATH)
        image_files = sorted([f for f in all_files if f.lower().endswith('.jpg')])
        processed_count = 0
        TARGET_WIDTH, TARGET_HEIGHT = 768, 1024

        for img_name in image_files:
            base_id = Path(img_name).stem
            # UPDATED: Validates and targets the matching mask inside the exact same flat directory
            expected_mask_name = f"{base_id}_mask.png"
            expected_mask_path = os.path.join(SRC_DATA_PATH, expected_mask_name)
            
            if os.path.exists(expected_mask_path):
                img_B = cv2.imread(os.path.join(SRC_DATA_PATH, img_name))
                img_B_resized = cv2.resize(img_B, (TARGET_WIDTH, TARGET_HEIGHT))
                
                mask_gray = cv2.imread(expected_mask_path, cv2.IMREAD_GRAYSCALE)
                img_A = mask_to_rgb(mask_gray)
                img_A = cv2.cvtColor(img_A, cv2.COLOR_RGB2BGR)  
                img_A_resized = cv2.resize(img_A, (TARGET_WIDTH, TARGET_HEIGHT))
                
                cv2.imwrite(os.path.join(TRAIN_A_DIR, f"{base_id}.png"), img_A_resized)
                cv2.imwrite(os.path.join(TRAIN_B_DIR, f"{base_id}.jpg"), img_B_resized)
                cv2.imwrite(os.path.join(TEST_A_DIR, f"{base_id}.png"), img_A_resized)
                processed_count += 1
        print(f"✅ Dataset ready! {processed_count} pairs generated.")
    else:
        print(f"❌ Error: Raw image folder '{SRC_DATA_PATH}' not found.")

os.chdir(REPO_DIR)

# ------------------------------------------------------------------------------
# STEP 4: Code Patching & Launching Pix2PixHD Training
# ------------------------------------------------------------------------------
print("\n--- STEP 4: Auto-patching legacy repository & launching training ---")

# Programmatic fix for modern Python compatibility (fractions.gcd -> math.gcd)
train_script_path = os.path.join(REPO_DIR, "train.py")
if os.path.exists(train_script_path):
    with open(train_script_path, "r") as f:
        code_content = f.read()
    
    if "fractions.gcd" in code_content:
        print("🔧 Legacy 'fractions.gcd' detected. Rewriting train.py to use modern 'math.gcd'...")
        # Inject standard math import if missing and swap out the broken gcd caller
        code_content = "import math\n" + code_content
        code_content = code_content.replace("fractions.gcd", "math.gcd")
        
        with open(train_script_path, "w") as f:
            f.write(code_content)
        print("✅ Hot-patch applied successfully.")
    else:
        print("✅ train.py code patch already applied.")

train_cmd = [
    "python", "train.py",
    "--dataroot", HD_DATASET_DIR,
    "--name", "palm_oil_hd_v1",
    "--label_nc", "0",
    "--no_instance",
    "--resize_or_crop", "none",
    "--no_html",
    
    # Model Configurations
    "--batchSize", "1",          
    "--niter", "100",            
    "--niter_decay", "100",      
    "--lr", "0.0002"             
]

print(f"Command Issued: {' '.join(train_cmd)}")
print("Stream initialized. Training output logs will display below in real-time...\n")

try:
    process = subprocess.Popen(
        train_cmd, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True, 
        bufsize=1,
        env=os.environ.copy()
    )
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
    process.stdout.close()
    train_return_code = process.wait()
    if train_return_code == 0:
        print("\n✅ Training complete! Checkpoints successfully saved.")
    else:
        print(f"\n❌ Training stopped early. Exit code: {train_return_code}")
except KeyboardInterrupt:
    print("\n🛑 Training manually interrupted by user.")
except Exception as e:
    print(f"\n❌ Pipeline Exception during training occurred: {e}")

# ------------------------------------------------------------------------------
# STEP 5: Launching Generation Test (Inference)
# ------------------------------------------------------------------------------
print("\n--- STEP 5: Generating High-Resolution Synthetic Images ---")

test_cmd = [
    "python", "test.py",
    "--dataroot", HD_DATASET_DIR,
    "--name", "palm_oil_hd_v1",
    "--label_nc", "0",
    "--no_instance",
    "--resize_or_crop", "none",
    "--how_many", "500"  
]

try:
    # Only run testing if the training run actually managed to output weight files
    checkpoint_file = os.path.join(REPO_DIR, "checkpoints", "palm_oil_hd_v1", "latest_net_G.pth")
    if not os.path.exists(checkpoint_file):
        print("❌ Cannot run generation step: Training checkpoints are missing because Step 4 crashed.")
    else:
        test_process = subprocess.Popen(
            test_cmd, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True, 
            bufsize=1,
            env=os.environ.copy()
        )
        for line in iter(test_process.stdout.readline, ""):
            print(line, end="")
        test_process.stdout.close()
        test_return_code = test_process.wait()
        
        if test_return_code == 0:
            print(f"\n========================================================================\n🎉 SUCCESS! PIPELINE COMPLETE 🎉\n\nSynthetic high-resolution (768x1024) images successfully generated at:\n{HOME_DIR}/pix2pixHD/results/palm_oil_hd_v1/test_latest/images/\n========================================================================\n")
except KeyboardInterrupt:
    print("\n🛑 Generation manually interrupted by user.")

--- STEP 1: Checking environment & packages ---
✅ Core dependencies already installed. Skipping installation phase.
PyTorch version: 2.11.0+cu128 | CUDA Available: True

--- STEP 2: Checking Pix2PixHD repository ---
Cloning codebase to: /home/project/26060809/RE8_PNG_Masks/pix2pixHD


Cloning into '/home/project/26060809/RE8_PNG_Masks/pix2pixHD'...



--- STEP 3: Checking dataset directory status ---
Dataset incomplete or missing. Reprocessing 768x1024 portrait targets...
✅ Dataset ready! 242 pairs generated.

--- STEP 4: Auto-patching legacy repository & launching training ---
🔧 Legacy 'fractions.gcd' detected. Rewriting train.py to use modern 'math.gcd'...
✅ Hot-patch applied successfully.
Command Issued: python train.py --dataroot /home/project/26060809/RE8_PNG_Masks/palm_oil_hd --name palm_oil_hd_v1 --label_nc 0 --no_instance --resize_or_crop none --no_html --batchSize 1 --niter 100 --niter_decay 100 --lr 0.0002
Stream initialized. Training output logs will display below in real-time...

------------ Options -------------
batchSize: 1
beta1: 0.5
checkpoints_dir: ./checkpoints
continue_train: False
data_type: 32
dataroot: /home/project/26060809/RE8_PNG_Masks/palm_oil_hd
debug: False
display_freq: 100
display_winsize: 512
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
input_nc: 3
instance_feat: False
isTrain: True
label_feat:

In [2]:
# ==============================================================================
# AUTOMATED FID SCORE EVALUATION (ISOLATED FUNCTION INJECTION METHOD)
# ==============================================================================
import os
import sys
import shutil
import subprocess

print("--- STEP 1: Ensuring pytorch-fid package is available ---")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pytorch-fid"])

# UPDATED: Reconfigured to match your new workspace pathing architecture
HOME_DIR = "/home/project/26060809/RE8_PNG_Masks"
REAL_IMAGES_DIR = os.path.join(HOME_DIR, "palm_oil_hd/train_B")  
SYNTH_IMAGES_DIR = os.path.join(HOME_DIR, "pix2pixHD/results/palm_oil_hd_v1/test_latest/images")
EVAL_SYNTH_DIR = os.path.join(HOME_DIR, "fid_eval_temp")

if not os.path.exists(REAL_IMAGES_DIR) or len(os.listdir(REAL_IMAGES_DIR)) == 0:
    print(f"❌ Error: Real image baseline directory missing at: {REAL_IMAGES_DIR}")
elif not os.path.exists(SYNTH_IMAGES_DIR) or len(os.listdir(SYNTH_IMAGES_DIR)) == 0:
    print(f"❌ Error: Synthetic image generation directory missing at: {SYNTH_IMAGES_DIR}")
else:
    print("\n--- STEP 2: Isolating production synthetic images from mask files ---")
    if os.path.exists(EVAL_SYNTH_DIR):
        shutil.rmtree(EVAL_SYNTH_DIR)
    os.makedirs(EVAL_SYNTH_DIR, exist_ok=True)
    
    copied_count = 0
    for file_name in os.listdir(SYNTH_IMAGES_DIR):
        if "synthesized_image" in file_name:
            shutil.copy(os.path.join(SYNTH_IMAGES_DIR, file_name), os.path.join(EVAL_SYNTH_DIR, file_name))
            copied_count += 1
            
    print(f"Extracted {copied_count} standalone synthetic images for evaluation.")
    print("\n--- STEP 3: Injecting Custom Frechet Math & Calculating FID ---")

    # Import the target library components
    import torch
    import numpy as np
    import scipy.linalg
    import pytorch_fid.fid_score
    from pytorch_fid.fid_score import calculate_fid_given_paths

    # 🔧 ISOLATED FUNCTION OVERRIDE: Replaces the broken internal math module natively
    def robust_calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
        mu1 = np.atleast_1d(mu1)
        mu2 = np.atleast_1d(mu2)
        sigma1 = np.atleast_2d(sigma1)
        sigma2 = np.atleast_2d(sigma2)

        diff = mu1 - mu2

        # Compute matrix square root without passing ANY problematic keywords
        sqrtm_res = scipy.linalg.sqrtm(sigma1.dot(sigma2))
        
        # Handle versions where sqrtm returns a tuple (matrix, errest) vs matrix only
        covmean = sqrtm_res[0] if isinstance(sqrtm_res, tuple) else sqrtm_res

        if not np.isfinite(covmean).all():
            offset = np.eye(sigma1.shape[0]) * eps
            sqrtm_res = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
            covmean = sqrtm_res[0] if isinstance(sqrtm_res, tuple) else sqrtm_res

        if np.iscomplexobj(covmean):
            tr_covmean = np.diagonal(covmean).real.sum()
        else:
            tr_covmean = np.trace(covmean)

        return (diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean)

    # Directly hot-swap the formula implementation within the package context
    pytorch_fid.fid_score.calculate_frechet_distance = robust_calculate_frechet_distance
    print("🔧 Isolated math architecture override injected safely.")

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Running feature vector distributions on device: {device}...")
    
    try:
        fid_value = calculate_fid_given_paths(
            paths=[REAL_IMAGES_DIR, EVAL_SYNTH_DIR],
            batch_size=50,
            device=device,
            dims=2048,
            num_workers=2
        )
        
        print("\n========================================================================")
        print(f"🎉 SUCCESS! Final calculated FID Score: {fid_value:.4f}")
        print("========================================================================")
        
    except Exception as e:
        print(f"\n❌ Execution breakdown occurred during math distribution pass: {e}")
    finally:
        # Cleanup temporary evaluation assets
        if os.path.exists(EVAL_SYNTH_DIR):
            shutil.rmtree(EVAL_SYNTH_DIR)
            print("🧹 Temporary calculation files purged successfully.")

--- STEP 1: Ensuring pytorch-fid package is available ---

--- STEP 2: Isolating production synthetic images from mask files ---
Extracted 242 standalone synthetic images for evaluation.

--- STEP 3: Injecting Custom Frechet Math & Calculating FID ---
🔧 Isolated math architecture override injected safely.
Running feature vector distributions on device: cuda:0...


100%|██████████| 5/5 [00:02<00:00,  1.84it/s]



🎉 SUCCESS! Final calculated FID Score: 25.3572
🧹 Temporary calculation files purged successfully.
